In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 03 · Preprocesamiento — de las decisiones a código con pruebas

> **Esqueleto.** Guía completa: `curso/kit-estudiante/guia-eda-y-preprocesamiento.md`, sección 4.

Cada decisión de la tabla del EDA se convierte en una **función pequeña** en
`src/proyecto/data/` y en **al menos una prueba** en `tests/`. Este notebook **solo orquesta**:
llama a las funciones y muestra resultados. No contiene lógica reutilizable.

Qué produce: datos limpios, **partición por la unidad de análisis** y una prueba que la verifica.

## 0. Cargar los datos crudos

In [2]:
import pandas as pd

from proyecto.data.load import load_raw_saber11

RAW_DIR = Path.cwd().resolve().parent / "data" / "raw"
# Se carga el archivo original tal cual; si no existe, se descarga de datos.gov.co (no se modifica).
# No se muestran filas antes de limpiar: el archivo trae estu_consecutivo y estu_fechanacimiento (datos personales).
df = load_raw_saber11(RAW_DIR)
print(f"datos originales: {df.shape[0]:,} filas x {df.shape[1]} columnas")

ID_COL = "cole_cod_dane_sede"   # sede del colegio: la entidad que se repite (decisión 10 del EDA)
TARGET = "punt_global"

datos originales: 144,766 filas x 51 columnas


## 1. Limpieza

Importa tus funciones de `src/proyecto/data/cleaning.py` (créalas con Claude a partir de tu tabla
de decisiones; renombra el paquete `proyecto` si quieres). **No modifiques `data/raw/`.**

In [3]:
try:
    from proyecto.data.cleaning import clean   # limpieza de la etapa 3, en src/proyecto/data/cleaning.py
except ModuleNotFoundError:
    clean = None
    print("No se pudo importar clean de src/proyecto/data/cleaning.py: ejecuta el notebook desde notebooks/.")

if df is None or clean is None:
    clean_df = None
else:
    clean_df = clean(df)
    print(f"antes: {df.shape}  ->  después: {clean_df.shape}")

antes: (144766, 51)  ->  después: (72148, 36)


## 2. Comprobaciones de la limpieza

Convierte tus decisiones en **afirmaciones** que fallan si algo no se cumple (no es un gráfico).

In [4]:
if clean_df is None:
    print("Sin limpieza todavía: esta celda se ejecutará cuando exista clean(df).")
else:
    from proyecto.data.cleaning import CONSTANT_COLUMNS, LEAKAGE_COLUMNS, PERSONAL_COLUMNS, SCHOOL_ID_COLUMNS

    # Cifras esperadas según reports/01_eda.md: 72.383 estudiantes sin duplicados - 235 de calendario «OTRO».
    checks = {
        "72.148 filas (sin duplicados ni calendario OTRO)": len(clean_df) == 72_148,
        "36 columnas": clean_df.shape[1] == 36,
        "sin datos personales": not set(PERSONAL_COLUMNS) & set(clean_df.columns),
        "sin columnas con fuga": not set(LEAKAGE_COLUMNS) & set(clean_df.columns),
        "sin columnas constantes": not set(CONSTANT_COLUMNS) & set(clean_df.columns),
        "sin identificadores del colegio (salvo la sede)": not set(SCHOOL_ID_COLUMNS) & set(clean_df.columns),
        "la sede se conserva para agrupar": ID_COL in clean_df.columns,
        "solo calendario A": set(clean_df["cole_calendario"]) == {"A"},
        "edad >= 13 o vacía": bool((clean_df["edad"].dropna() >= 13).all()),
        "sin_estrato solo 0/1": set(clean_df["sin_estrato"]) <= {0, 1},
        "estrato 1-6 o vacío (nunca 0)": set(clean_df["fami_estratovivienda"].dropna()) <= set(range(1, 7)),
        "personas 1-5 o vacío": set(clean_df["fami_personashogar"].dropna()) <= set(range(1, 6)),
        "cuartos 1-6 o vacío": set(clean_df["fami_cuartoshogar"].dropna()) <= set(range(1, 7)),
        "objetivo sin vacíos": clean_df[TARGET].notna().all(),
    }
    result = pd.Series(checks, name="se cumple")
    assert result.all(), result[~result]
    print(result)

72.148 filas (sin duplicados ni calendario OTRO)    True
36 columnas                                         True
sin datos personales                                True
sin columnas con fuga                               True
sin columnas constantes                             True
sin identificadores del colegio (salvo la sede)     True
la sede se conserva para agrupar                    True
solo calendario A                                   True
edad >= 13 o vacía                                  True
sin_estrato solo 0/1                                True
estrato 1-6 o vacío (nunca 0)                       True
personas 1-5 o vacío                                True
cuartos 1-6 o vacío                                 True
objetivo sin vacíos                                 True
Name: se cumple, dtype: bool


## 3. Partición por la unidad de análisis

Si una entidad aparece en varias filas, **todas sus filas van al mismo lado**. Importa tu función
de `src/proyecto/data/split.py` (modelo en la sección 4.3 de la guía) y **verifica** que ninguna
entidad quede en los dos lados.

In [5]:
try:
    from proyecto.data.split import split_by_group
except ModuleNotFoundError:
    split_by_group = None
    print("No se pudo importar split_by_group de src/proyecto/data/split.py: ejecuta el notebook desde notebooks/.")

if clean_df is None or split_by_group is None or ID_COL is None:
    train = test = None
    print("Pendiente: necesitas datos limpios, split_by_group y ID_COL definido.")
else:
    # 20 % de las sedes para prueba, semilla 42 (especificaciones/03-preprocesamiento.md, sección 1).
    train, test = split_by_group(clean_df, group_col=ID_COL, test_size=0.2, seed=42)
    assert set(train[ID_COL]).isdisjoint(set(test[ID_COL])), "hay sedes en los dos lados"
    assert len(train) + len(test) == len(clean_df), "se perdieron filas al partir"
    compartidas = len(set(train[ID_COL]) & set(test[ID_COL]))
    print(f"entrenamiento: {len(train):,} filas · {train[ID_COL].nunique():,} sedes")
    print(f"prueba:        {len(test):,} filas · {test[ID_COL].nunique():,} sedes (cerrada hasta la etapa 7)")
    print(f"sedes en los dos lados: {compartidas}")

entrenamiento: 57,286 filas · 1,155 sedes
prueba:        14,862 filas · 289 sedes (cerrada hasta la etapa 7)
sedes en los dos lados: 0


**Mi unidad de análisis y por qué parto así:** una fila es un estudiante que presentó Saber 11 en el periodo 20224 en un
colegio de Antioquia (calendario A). El estudiante no se repite tras quitar duplicados, pero la **sede del colegio**
(`cole_cod_dane_sede`) agrupa a muchos estudiantes y sola explica el 36 % de la variación del puntaje. La app recibirá
colegios que el modelo no vio, así que se parte **por sede**: 20 % de las sedes para prueba con `GroupShuffleSplit`
(semilla 42), y la validación cruzada de las etapas siguientes también agrupa por sede (`GroupKFold`). Partir por
estudiante daría un MAE optimista.

## 4. Guardar los datos de la etapa

Se guardan en `data/processed/` (no se versionan: se regeneran con este notebook). La prueba **no
se vuelve a mirar** hasta la validación final (etapa 7).

In [6]:
if train is None or test is None:
    print("Sin partición todavía: esta celda guardará los datos cuando exista.")
else:
    import joblib
    out = Path.cwd().resolve().parent / "data" / "processed"
    out.mkdir(parents=True, exist_ok=True)
    joblib.dump(train, out / "train.joblib")
    joblib.dump(test, out / "test.joblib")
    print("guardado en data/processed/")

guardado en data/processed/


## 5. Las pruebas automáticas

Desde la raíz del repo:

```bash
pytest -q
```

Debe existir **al menos una prueba por decisión** (`tests/test_cleaning.py`) y la prueba de la
partición (`tests/test_split.py`, modelo en la sección 4.3 de la guía).

## Entregable de esta etapa (hito H1)

- [x] La limpieza corre de principio a fin desde el repo.
- [x] La partición respeta mi unidad de análisis y está **verificada con una prueba**.
- [x] `pytest` en verde.
- [x] `data/raw/` no cambió.
- [x] Declaré mi unidad de análisis por escrito.